# Simple UDP Communications

- UDP
    - Connectionless, don't call `connect()`
    - Server uses `bind()`
    - Server and Client uses `sendto()` or `recvfrom()`
- Multicast
    - Sends a message to a group of computers on a subnet who subscribe
 
# References

- BearSnacks
    - [C Socket Programming](https://github.com/Bear-Snacks/bear-snacks.github.io/blob/main/c/socket-programming.md)
    - [C Multicast Programming](https://github.com/Bear-Snacks/bear-snacks.github.io/blob/main/c/c-multicast.md)
- [Multicast How Too](https://tldp.org/HOWTO/Multicast-HOWTO-6.html)
- [IBM: IP Multicasting](https://www.ibm.com/support/knowledgecenter/ssw_ibm_i_71/rzab6/cmulticast.htm)
- [IBM: Example](https://www.ibm.com/support/knowledgecenter/ssw_ibm_i_71/rzab6/xmulticast.htm)
- [IBM: Send Example](https://www.ibm.com/support/knowledgecenter/ssw_ibm_i_71/rzab6/x1multicast.htm)
- [gunther](https://github.com/gecko-robotics/gunther)
- [mcastsocket](https://github.com/mcfletch/mcastsocket)

In [27]:
import threading
import socket
import time
import os
import struct

## UDP Socket Communications

In [37]:
event = threading.Event()
event.set()

# host = socket.gethostbyname(socket.gethostname())
host = "0.0.0.0"
addr = (host, 5550,)

In [38]:
def server(e,addr):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        sock.settimeout(0.01)
        sock.bind(addr)
        # sock.listen(0)
    except Exception as ex:
        print(ex)
        e.clear()
    
    for i in range(5):
        try:
            data, address = sock.recvfrom(32)
        except socket.timeout:
            time.sleep(0.1)
            if not e.is_set():
                sock.close()
                return
            continue
        # except ConnectionRefusedError:
        #     a,p = self.sock.getpeername()
        #     print(f"*** ConnectionRefusedError {a}:{p} ***")
        #     sock.close()
        #     return
            
        sock.sendto(b'Hello back!', address)
        time.sleep(0.1)

    sock.close()
    e.clear()


def client(e,addr):
    msgs = 0

    # print('Receiving messages...')

    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.settimeout(0.01)
        # sock.connect(addr)
    except Exception as ex:
        print(ex)
        e.clear()
    
    while e.is_set():
        sock.sendto(b"hello", addr)

        try:
            data, address = sock.recvfrom(32)

            msgs += 1
            print(f"Messages recieved[{msgs}] from {address}: {data}", end="\r")
        
        except socket.timeout:
            time.sleep(0.1)
        # except ConnectionRefusedError:
        #     a,p = self.sock.getpeername()
        #     print(f"*** ConnectionRefusedError {a}:{p} ***")
        #     sock.close()
        #     return

    sock.close()

    print(f'\nTotal received {msgs} messages.')

In [39]:
event.set()

s = threading.Thread(target=server, args=(event,addr,),name="server")
s.start()
print('Started {}'.format(s.name))

c = threading.Thread(target=client, args=(event,addr,), name="client")
c.start()
print('Started {}'.format(c.name))

s.join()
c.join()

for p in [s, c]:
    print('{} is alive: {}'.format(p.name, p.is_alive()))

Started server
Started client
Messages recieved[5] from ('127.0.0.1', 5550): b'Hello back!'
Total received 5 messages.
server is alive: False
client is alive: False


## Multicast

![](https://www.ibm.com/support/knowledgecenter/ssw_ibm_i_71/rzab6/rzab6507.gif)

```
  0                           31            Address Range:
 +-+----------------------------+
 |0|       Class A Address      |       0.0.0.0 - 127.255.255.255
 +-+----------------------------+
 +-+-+--------------------------+
 |1 0|     Class B Address      |     128.0.0.0 - 191.255.255.255
 +-+-+--------------------------+
 +-+-+-+------------------------+
 |1 1 0|   Class C Address      |     192.0.0.0 - 223.255.255.255
 +-+-+-+------------------------+
 +-+-+-+-+----------------------+
 |1 1 1 0|  MULTICAST Address   |     224.0.0.0 - 239.255.255.255
 +-+-+-+-+----------------------+
 +-+-+-+-+-+--------------------+
 |1 1 1 1 0|     Reserved       |     240.0.0.0 - 247.255.255.255
 +-+-+-+-+-+--------------------+
```

IPv4 range: 224.0.0.1 to 239.255.255.255

- `IP_ADD_MEMBERSHIP`: Joins the multicast group specified
- `IP_DROP_MEMBERSHIP`: Leaves the multicast group specified
- `IP_MULTICAST_IF`: Sets the interface over which outgoing multicast datagrams should be sent
- `IP_MULTICAST_TTL`: Sets the Time To Live (TTL) in the IP header for outgoing multicast datagrams
- `IP_MULTICAST_LOOP`: Specifies whether a copy of an outgoing multicast datagram should be
delivered to the sending host as long as it is a member of the multicast group. **This is enabled
by default** and must be turned off if you don't want to see your messages sent back to your
process

TTL | Scope
---:|------------------------------------------------------------------
   0| Restricted to the same host. Won't be output by any interface.
   1| Restricted to the same subnet. Won't be forwarded by a router.
 <32| Restricted to the same site, organization or department.
 <64| Restricted to the same region.
<128| Restricted to the same continent.
<255| Unrestricted in scope. Global.

The setsockopt() API also accepts the following IPPROTO_IPv6 level flags:

- `IPv6_MULTICAST_IF`: Sets the interface over which outgoing multicast datagrams are sent
- `IPv6_MULTICAST_HOPS`: Sets the hop limit values that are used for subsequent multicast
packets sent by a socket
- `IPv6_MULTICAST_LOOP`: Specifies whether a copy of an outgoing multicast datagram should
  be delivered to the sending host as long as it is a member of the multicast group
- `IPv6_JOIN_GROUP`: Joins the multicast group specified
- `IPv6_LEAVE_GROUP`: Leaves the multicast group specified

So this is used by Apple's Bonjour or mDNS for apps to announce their services on a network. You can listen in and do passive network mapping. However you will get responses like this:

```
Got b'\x00\x00\x00\x00\x00\x02\x00\x00\x00\x03\x00\x009a0:fb:c5:3d:96:57@fe80::a2fb:c5ff:fe3d:9657-support' from ('10.0.1.25', 5353)
Got b'\x00\x00\x84\x00\x00\x00\x00\x02\x00\x00\x00\x02\x016\x01C\x01F\x01F\x01D\x01D\x01C\x014\x01F\x019\x019\x01C\x015\x01A\x018\x011\x010\x010\x010\x010\x010\x010\x010\x010\x010\x010' from ('10.0.1.25', 5353)
Got b'\x00\x00\x00\x00\x00\x03\x00\x04\x00\x00\x00\x00\x0f_companion-link\x04_tcp\x05local\x00\x00\x0c\x00\x01\x07_rdlink\xc0\x1c\x00\x0c\x00\x01\x08_home' from ('10.0.1.25', 5353)
Got b'\x00\x00\x84\x00\x00\x00\x00\x07\x00\x00\x00\x029a0:fb:c5:3d:96:57@fe80::a2fb:c5ff:fe3d:9657-support' from ('10.0.1.25', 5353)
timeout
timeout
Got b'\x00\x00\x84\x00\x00\x00\x00\x07\x00\x00\x00\x029a0:fb:c5:3d:96:57@fe80::a2fb:c5ff:fe3d:9657-support' from ('10.0.1.25', 5353)
timeout
Got b'\x00\x00\x84\x00\x00\x00\x00\x02\x00\x00\x00\x02\x016\x01C\x01F\x01F\x01D\x01D\x01C\x014\x01F\x019\x019\x01C\x015\x01A\x018\x011\x010\x010\x010\x010\x010\x010\x010\x010\x010\x010' from ('10.0.1.25', 5353)
timeout
```

In [50]:
def mdns():
    # Bonjour/mDNS
    mcast_addr = "224.0.0.251"
    mcast_port = 5353
    sock = socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    
    # sock.setblocking(1)
    sock.settimeout(0.5)

    # set TTL (time to live) on packets
    ttl = struct.pack('b', 1)
    sock.setsockopt(socket.IPPROTO_IP, socket.IP_MULTICAST_TTL, ttl)
    
    # disable (0) or enable (1) loopback support, enabling allows
    # the local machine to see its own packets
    sock.setsockopt(socket.SOL_IP, socket.IP_MULTICAST_LOOP, 1)

    # Restrict multicast operation to the given interface/ip (instead of using routing)
    #
    # Sets the IP_MULTICAST_IF option on the socket to restrict multicast
    # operations to a particular interface.  This is done without reference
    # to the system routing tables, so you do not need to set up a 224.0.0.0/4
    # route on the system to receive multicast on a given interface if you 
    # have bound the socket to anything other than ('',port) or (group,port).
    #
    # Note: for ipv6 sockets you must specify the interface name, rather than
    # the interface IP address, as the API uses interface ids to specify the 
    # interface to which to limit.
    #
    # Set the multicast interface
    multicast_if_addr = "0.0.0.0"  # Replace with your interface address
    sock.setsockopt(socket.IPPROTO_IP, socket.IP_MULTICAST_IF, socket.inet_aton(multicast_if_addr))

    # allow multiple connections
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)

    try:
        # Note: multicast is *not* working if we don't bind on all interfaces, most likely
        # because the 224.* isn't getting mapped (routed) to the address of the interface...
        # to debug that case, see if {{{ip route add 224.0.0.0/4 dev br0}}} (or whatever your
        # interface is) makes the route suddenly start working...
        # However, if you want to bind on just one interface, use the group
        
        # sock.bind((mcast_addr, mcast_port))
        sock.bind(("0.0.0.0", mcast_port))
    except OSError as e:
        print("*** {} ***".format(e))
        return

    mreq = struct.pack("=4sl", socket.inet_aton(mcast_addr), socket.INADDR_ANY)
    sock.setsockopt(socket.IPPROTO_IP, socket.IP_ADD_MEMBERSHIP, mreq)

    for i in range(10):
        try:
            data, address = sock.recvfrom(64)
            print(f"Got {data} from {address}")
        except socket.timeout:
            print("timeout")
            continue
        except Exception as e:
            print(f"{e}")
            continue

    sock.setsockopt(socket.SOL_IP, socket.IP_DROP_MEMBERSHIP, mreq)
    sock.close()

mdns()

timeout
Got b'\x00\x00\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x07_rdlink\x04_tcp\x05local\x00\x00\x0c\x80\x01\x0f_companion-link\xc0\x14\x00\x0c\x80\x01\x08_home' from ('10.0.1.97', 5353)
timeout
timeout
Got b'\x00\x00\x00\x00\x00\x04\x00\x02\x00\x00\x00\x00\x07_rdlink\x04_tcp\x05local\x00\x00\x0c\x00\x01\x0f_companion-link\xc0\x14\x00\x0c\x00\x01\x08_home' from ('10.0.1.97', 5353)
timeout
timeout
timeout
timeout
timeout
